# Hyperparameter Tuning of Individual Recommender Models

This notebook performs hyperparameter tuning for the individual recommender models used in the project.

The models are evaluated on the validation set using Top-10 ranking metrics. The best configuration is selected according to **MRR@10**.

The following models can be tuned:

- BPR
- FISM
- ItemKNN
- UserKNN
- SLIMElastic
- EASE

Results from every experiment are saved immediately to a CSV file so that completed experiments do not need to be repeated if execution is interrupted.

## Code cell — imports

In [1]:
import csv
import itertools
import os
import pandas as pd

from recbole.quick_start import run_recbole

## Code cell — settings

In [2]:
DATASET = "ml-100k"

# Choose model to tune:
# BPR, FISM, ItemKNN, UserKNN, SLIMElastic, EASE
MODEL_TO_TUNE = "EASE"

OUTPUT_FILE = "tuning_results.csv"

## Hyperparameter Search Spaces

A grid search is used to evaluate different hyperparameter configurations for each recommender.

Each configuration is trained using the same dataset split and evaluation procedure, allowing the models and parameter settings to be compared consistently.

In [3]:
MODELS = {
    "BPR": {
        "recbole_model": "BPR",
        "grid": {
            "embedding_size": [32, 64, 128],
            "learning_rate": [0.0005, 0.001, 0.005],
        },
    },

    "FISM": {
        "recbole_model": "FISM",
        "grid": {
            "embedding_size": [32, 64, 128],
            "learning_rate": [0.0005, 0.001, 0.005],
            "reg_weights": [
                [0.001, 0.001],
                [0.01, 0.01],
                [0.1, 0.1],
            ],
        },
    },

    "ItemKNN": {
        "recbole_model": "ItemKNN",
        "grid": {
            "k": [20, 50, 100, 200],
        },
    },

    "UserKNN": {
        "recbole_model": "ItemKNN",
        "grid": {
            "k": [20, 50, 100, 200],
            "knn_method": ["user"],
        },
    },

    "SLIMElastic": {
        "recbole_model": "SLIMElastic",
        "grid": {
            "alpha": [0.01, 0.1, 0.2, 0.5],
            "l1_ratio": [0.01, 0.02, 0.1, 0.5],
        },
    },

    "EASE": {
        "recbole_model": "EASE",
        "grid": {
            "reg_weight": [
                50.0,
                100.0,
                250.0,
                500.0,
                1000.0,
            ],
        },
    },
}

In [4]:
def generate_combinations(grid):
    keys = list(grid.keys())
    values = list(grid.values())

    for combination in itertools.product(*values):
        yield dict(zip(keys, combination))

def load_completed_experiments(model_name):
    completed = set()

    if not os.path.exists(OUTPUT_FILE):
        return completed

    with open(OUTPUT_FILE, "r", newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            if row["model"] == model_name:
                completed.add(row["parameters"])

    return completed

def save_result(row):
    file_exists = os.path.exists(OUTPUT_FILE)

    fieldnames = [
        "model",
        "parameters",
        "valid_recall@10",
        "valid_mrr@10",
        "valid_ndcg@10",
        "valid_hit@10",
        "valid_precision@10",
    ]

    with open(OUTPUT_FILE, "a", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames
        )

        if not file_exists:
            writer.writeheader()

        writer.writerow(row)


def load_completed_experiments(model_name):
    completed = set()

    if not os.path.exists(OUTPUT_FILE):
        return completed

    with open(OUTPUT_FILE, "r", newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            if row["model"] == model_name:
                completed.add(row["parameters"])

    return completed

In [5]:
def tune_model(model_name):

    if model_name not in MODELS:
        raise ValueError(f"Unknown model: {model_name}")

    completed = load_completed_experiments(model_name)

    print(f"Already completed: {len(completed)}")

    settings = MODELS[model_name]

    recbole_model = settings["recbole_model"]
    grid = settings["grid"]

    combinations = list(generate_combinations(grid))

    print("\n" + "=" * 70)
    print(f"TUNING MODEL: {model_name}")
    print(f"Number of experiments: {len(combinations)}")
    print("=" * 70)

    results = []

    for experiment_number, params in enumerate(
        combinations,
        start=1
    ):
        params_string = str(params)

        # Skip experiments already stored in CSV
        if params_string in completed:
            print(f"Skipping completed experiment: {params}")
            continue

        print("\n" + "-" * 70)
        print(
            f"{model_name} Experiment "
            f"{experiment_number}/{len(combinations)}"
        )
        print("-" * 70)

        print("Parameters:")
        print(params)

        try:

            result = run_recbole(
                model=recbole_model,
                dataset=DATASET,
                config_dict=params,
                saved=False
            )

            valid = result["best_valid_result"]

            row = {
                "model": model_name,
                "parameters": str(params),

                "valid_recall@10": float(
                    valid["recall@10"]
                ),

                "valid_mrr@10": float(
                    valid["mrr@10"]
                ),

                "valid_ndcg@10": float(
                    valid["ndcg@10"]
                ),

                "valid_hit@10": float(
                    valid["hit@10"]
                ),

                "valid_precision@10": float(
                    valid["precision@10"]
                ),
            }

            results.append(row)

            # Save after every successful experiment
            save_result(row)

            print("\nVALIDATION RESULT")
            print(
                f"NDCG@10: "
                f"{row['valid_ndcg@10']:.4f}"
            )

        except Exception as error:

            print("\nERROR")
            print(
                f"Experiment failed: {params}"
            )
            print(error)

            continue

    # ---------------------------------------------------------
    # Load ALL results from CSV
    # ---------------------------------------------------------

    if not os.path.exists(OUTPUT_FILE):
        print(f"\nNo successful experiments for {model_name}.")
        return None

    all_results = pd.read_csv(OUTPUT_FILE)

    model_results = all_results[
        all_results["model"] == model_name
    ].copy()

    if model_results.empty:
        print(f"\nNo successful experiments for {model_name}.")
        return None

    # ---------------------------------------------------------
    # Find best configuration by NDCG@10
    # ---------------------------------------------------------

    best_idx = model_results[
        "valid_ndcg@10"
    ].idxmax()

    best = model_results.loc[best_idx]

    # ---------------------------------------------------------
    # Print best configuration
    # ---------------------------------------------------------

    print("\n" + "=" * 70)
    print(f"BEST {model_name} CONFIGURATION")
    print("=" * 70)

    print("Parameters:")
    print(best["parameters"])

    print("\nValidation metrics:")

    print(
        f"NDCG@10:      "
        f"{best['valid_ndcg@10']:.4f}"
    )

    print(
        f"Recall@10:    "
        f"{best['valid_recall@10']:.4f}"
    )

    print(
        f"MRR@10:       "
        f"{best['valid_mrr@10']:.4f}"
    )

    print(
        f"Hit@10:       "
        f"{best['valid_hit@10']:.4f}"
    )

    print(
        f"Precision@10: "
        f"{best['valid_precision@10']:.4f}"
    )

    return best

In [6]:
best_bpr = tune_model("BPR")

Already completed: 9

TUNING MODEL: BPR
Number of experiments: 9
Skipping completed experiment: {'embedding_size': 32, 'learning_rate': 0.0005}
Skipping completed experiment: {'embedding_size': 32, 'learning_rate': 0.001}
Skipping completed experiment: {'embedding_size': 32, 'learning_rate': 0.005}
Skipping completed experiment: {'embedding_size': 64, 'learning_rate': 0.0005}
Skipping completed experiment: {'embedding_size': 64, 'learning_rate': 0.001}
Skipping completed experiment: {'embedding_size': 64, 'learning_rate': 0.005}
Skipping completed experiment: {'embedding_size': 128, 'learning_rate': 0.0005}
Skipping completed experiment: {'embedding_size': 128, 'learning_rate': 0.001}
Skipping completed experiment: {'embedding_size': 128, 'learning_rate': 0.005}

BEST BPR CONFIGURATION
Parameters:
{'embedding_size': 128, 'learning_rate': 0.001}

Validation metrics:
NDCG@10:      0.2387
Recall@10:    0.2193
MRR@10:       0.4018
Hit@10:       0.7625
Precision@10: 0.1589


In [ ]:
best_fism = tune_model("FISM")

07 Oct 21:59    INFO  ['/Users/randy/Desktop/Recommender Systems/Final Project/.venv_recbole/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v357c8491518bb2d858c36ba0129beef5cd1934f41.json']
07 Oct 21:59    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/Final Project/.venv_recbole/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.0005
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place =

Already completed: 0

TUNING MODEL: FISM
Number of experiments: 27

----------------------------------------------------------------------
FISM Experiment 1/27
----------------------------------------------------------------------
Parameters:
{'embedding_size': 32, 'learning_rate': 0.0005, 'reg_weights': [0.001, 0.001]}


 None
NEG_PREFIX = neg_
load_col = {'inter': ['user_id', 'item_id', 'rating', 'timestamp']}
unload_col = None
unused_col = None
additional_feat_suffix = None
rm_dup_inter = None
val_interval = None
filter_inter_by_user_or_item = True
user_inter_num_interval = None
item_inter_num_interval = None
alias_of_user_id = None
alias_of_item_id = None
alias_of_entity_id = None
alias_of_relation_id = None
preload_weight = None
normalize_field = None
normalize_all = True
ITEM_LIST_LENGTH_FIELD = item_length
LIST_SUFFIX = _list
MAX_ITEM_LIST_LENGTH = 50
POSITION_FIELD = position_id
HEAD_ENTITY_ID_FIELD = head_id
TAIL_ENTITY_ID_FIELD = tail_id
RELATION_ID_FIELD = relation_id
ENTITY_ID_FIELD = entity_id
kg_reverse_r = False
entity_kg_num_interval = None
relation_kg_num_interval = None
benchmark_filename = None

Other Hyper Parameters: 
worker = 0
wandb_project = recbole
shuffle = True
require_pow = False
enable_amp = False
enable_scaler = False
transform = None
embedding_size = 32
split_to = 0
reg_we


VALIDATION RESULT
NDCG@10: 0.1656

----------------------------------------------------------------------
FISM Experiment 2/27
----------------------------------------------------------------------
Parameters:
{'embedding_size': 32, 'learning_rate': 0.0005, 'reg_weights': [0.01, 0.01]}


07 Oct 22:04    INFO  ['/Users/randy/Desktop/Recommender Systems/Final Project/.venv_recbole/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v357c8491518bb2d858c36ba0129beef5cd1934f41.json']
07 Oct 22:04    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/Final Project/.venv_recbole/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.0005
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place =


VALIDATION RESULT
NDCG@10: 0.1213

----------------------------------------------------------------------
FISM Experiment 3/27
----------------------------------------------------------------------
Parameters:
{'embedding_size': 32, 'learning_rate': 0.0005, 'reg_weights': [0.1, 0.1]}


07 Oct 22:09    INFO  ['/Users/randy/Desktop/Recommender Systems/Final Project/.venv_recbole/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v357c8491518bb2d858c36ba0129beef5cd1934f41.json']
07 Oct 22:09    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/Final Project/.venv_recbole/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.0005
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place =

In [20]:
best_itemknn = tune_model("ItemKNN")

Already completed: 0

TUNING MODEL: ItemKNN
Number of experiments: 4

----------------------------------------------------------------------
ItemKNN Experiment 1/4
----------------------------------------------------------------------
Parameters:
{'k': 20}


07 Oct 21:56    INFO  ['/Users/randy/Desktop/Recommender Systems/Final Project/.venv_recbole/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v3933c33735439af6695cebc3fc66f87ee410fa302.json']
07 Oct 21:56    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/Final Project/.venv_recbole/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 


VALIDATION RESULT
NDCG@10: 0.2063

----------------------------------------------------------------------
ItemKNN Experiment 2/4
----------------------------------------------------------------------
Parameters:
{'k': 50}


07 Oct 21:56    INFO  ['/Users/randy/Desktop/Recommender Systems/Final Project/.venv_recbole/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v3933c33735439af6695cebc3fc66f87ee410fa302.json']
07 Oct 21:56    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/Final Project/.venv_recbole/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 


VALIDATION RESULT
NDCG@10: 0.2163

----------------------------------------------------------------------
ItemKNN Experiment 3/4
----------------------------------------------------------------------
Parameters:
{'k': 100}


07 Oct 21:56    INFO  ['/Users/randy/Desktop/Recommender Systems/Final Project/.venv_recbole/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v3933c33735439af6695cebc3fc66f87ee410fa302.json']
07 Oct 21:56    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/Final Project/.venv_recbole/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 


VALIDATION RESULT
NDCG@10: 0.2213

----------------------------------------------------------------------
ItemKNN Experiment 4/4
----------------------------------------------------------------------
Parameters:
{'k': 200}


07 Oct 21:56    INFO  ['/Users/randy/Desktop/Recommender Systems/Final Project/.venv_recbole/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v3933c33735439af6695cebc3fc66f87ee410fa302.json']
07 Oct 21:56    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/Final Project/.venv_recbole/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 


VALIDATION RESULT
NDCG@10: 0.2230

BEST ItemKNN CONFIGURATION
Parameters:
{'k': 200}

Validation metrics:
NDCG@10:      0.2230
Recall@10:    0.2073
MRR@10:       0.3706
Hit@10:       0.7328
Precision@10: 0.1550


In [21]:
best_userknn = tune_model("UserKNN")

Already completed: 0

TUNING MODEL: UserKNN
Number of experiments: 4

----------------------------------------------------------------------
UserKNN Experiment 1/4
----------------------------------------------------------------------
Parameters:
{'k': 20}


07 Oct 21:56    INFO  ['/Users/randy/Desktop/Recommender Systems/Final Project/.venv_recbole/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v3933c33735439af6695cebc3fc66f87ee410fa302.json']
07 Oct 21:56    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/Final Project/.venv_recbole/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 


VALIDATION RESULT
NDCG@10: 0.2063

----------------------------------------------------------------------
UserKNN Experiment 2/4
----------------------------------------------------------------------
Parameters:
{'k': 50}


07 Oct 21:56    INFO  ['/Users/randy/Desktop/Recommender Systems/Final Project/.venv_recbole/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v3933c33735439af6695cebc3fc66f87ee410fa302.json']
07 Oct 21:56    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/Final Project/.venv_recbole/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 


VALIDATION RESULT
NDCG@10: 0.2163

----------------------------------------------------------------------
UserKNN Experiment 3/4
----------------------------------------------------------------------
Parameters:
{'k': 100}


07 Oct 21:56    INFO  ['/Users/randy/Desktop/Recommender Systems/Final Project/.venv_recbole/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v3933c33735439af6695cebc3fc66f87ee410fa302.json']
07 Oct 21:56    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/Final Project/.venv_recbole/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 


VALIDATION RESULT
NDCG@10: 0.2213

----------------------------------------------------------------------
UserKNN Experiment 4/4
----------------------------------------------------------------------
Parameters:
{'k': 200}


07 Oct 21:56    INFO  ['/Users/randy/Desktop/Recommender Systems/Final Project/.venv_recbole/lib/python3.10/site-packages/ipykernel_launcher.py', '--f=/Users/randy/Library/Jupyter/runtime/kernel-v3933c33735439af6695cebc3fc66f87ee410fa302.json']
07 Oct 21:56    INFO  
General Hyper Parameters:
gpu_id = 0
use_gpu = True
seed = 2020
state = INFO
reproducibility = True
data_path = /Users/randy/Desktop/Recommender Systems/Final Project/.venv_recbole/lib/python3.10/site-packages/recbole/config/../dataset_example/ml-100k
checkpoint_dir = saved
show_progress = True
save_dataset = False
dataset_save_path = None
save_dataloaders = False
dataloaders_save_path = None
log_wandb = False

Training Hyper Parameters:
epochs = 300
train_batch_size = 2048
learner = adam
learning_rate = 0.001
train_neg_sample_args = {'distribution': 'uniform', 'sample_num': 1, 'alpha': 1.0, 'dynamic': False, 'candidate_num': 0}
eval_step = 1
stopping_step = 10
clip_grad_norm = None
weight_decay = 0.0
loss_decimal_place = 


VALIDATION RESULT
NDCG@10: 0.2230

BEST UserKNN CONFIGURATION
Parameters:
{'k': 200}

Validation metrics:
NDCG@10:      0.2230
Recall@10:    0.2073
MRR@10:       0.3706
Hit@10:       0.7328
Precision@10: 0.1550


In [ ]:
best_slim = tune_model("SLIMElastic")

In [ ]:
best_ease = tune_model("EASE")

In [22]:
results_df = pd.read_csv(OUTPUT_FILE)

results_df

,model,parameters,valid_recall@10,valid_mrr@10,valid_ndcg@10,valid_hit@10,valid_precision@10
0,BPR,"{'embedding_size': 32, 'learning_rate': 0.0005}",0.2039,0.3745,0.2249,0.7306,0.1592
1,BPR,"{'embedding_size': 32, 'learning_rate': 0.001}",0.2093,0.3756,0.2283,0.7423,0.1612
2,BPR,"{'embedding_size': 32, 'learning_rate': 0.005}",0.1979,0.3648,0.2162,0.7126,0.1507
3,BPR,"{'embedding_size': 64, 'learning_rate': 0.0005}",0.2005,0.3853,0.2260,0.7211,0.1546
4,BPR,"{'embedding_size': 64, 'learning_rate': 0.001}",0.2085,0.3893,0.2302,0.7402,0.1583
5,BPR,"{'embedding_size': 64, 'learning_rate': 0.005}",0.2027,0.3750,0.2233,0.7370,0.1552
6,BPR,"{'embedding_size': 128, 'learning_rate': 0.0005}",0.2203,0.3923,0.2382,0.7593,0.1655
7,BPR,"{'embedding_size': 128, 'learning_rate': 0.001}",0.2193,0.4018,0.2387,0.7625,0.1589
8,BPR,"{'embedding_size': 128, 'learning_rate': 0.005}",0.1985,0.3665,0.2160,0.7200,0.1487
9,ItemKNN,{'k': 20},0.1884,0.3553,0.2063,0.7041,0.1471
